# Healthcare Data Governance and Cleaning

**Demonstration project using a fabricated synthetic dataset.** The original CSV shown in the task page was not available, so this dataset is created solely to demonstrate the workflow. It must not be represented as the task-provided dataset or real patient data.

## Governance boundary
Clinical measurements are not changed, imputed, or judged against clinical thresholds. Clinical range rules require an approved specification with units. Duplicate patient IDs are flagged, not automatically deleted, because they may represent multiple encounters.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path("..")
INPUT_FILE = ROOT / "data" / "raw" / "synthetic_demo_patient_records.csv"
OUT = ROOT / "data"
raw = pd.read_csv(INPUT_FILE, dtype={"patient_id":"string","postal_code":"string"})
df = raw.copy(deep=True)
print(f"Rows: {len(raw)} | Columns: {len(raw.columns)}")
display(raw.head())


## 1. Inspect data types and completeness

In [ ]:
profile = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean()*100).round(2),
    "unique_values": df.nunique(dropna=True)
})
display(profile)
profile.to_csv(OUT / "quality_profile.csv")


## 2. Apply conservative formatting rules

In [ ]:
# Trim whitespace and convert blank strings to missing values.
for col in df.select_dtypes(include=["object","string"]).columns:
    df[col] = df[col].astype("string").str.strip().replace("", pd.NA)

# Parse dates. Failed values become missing in the cleaned copy and are logged.
original_dates = raw["visit_date"].copy()
parsed_dates = pd.to_datetime(df["visit_date"], errors="coerce")
date_failures = df["visit_date"].notna() & parsed_dates.isna()
df["visit_date"] = parsed_dates.dt.strftime("%Y-%m-%d").astype("string")
df.loc[date_failures, "visit_date"] = pd.NA


## 3. Validate missing values and duplicate identifiers

In [ ]:
exceptions = []
def log_issue(row_index, field, issue, rule, severity, action):
    exceptions.append({
        "source_row_number": int(row_index)+2,
        "field_name": field,
        "issue_type": issue,
        "validation_rule": rule,
        "severity": severity,
        "recommended_action": action,
        "status": "open - review"
    })

for i, row in raw.iterrows():
    for field in ["age_years","sex","visit_date","systolic_bp_mmHg","diastolic_bp_mmHg"]:
        if pd.isna(row[field]) or (isinstance(row[field], str) and not row[field].strip()):
            log_issue(i, field, "missing_value", "Review missingness against source specification",
                      "review", "Check source; do not infer or invent a value")
    if pd.notna(original_dates.iloc[i]) and pd.isna(pd.to_datetime(original_dates.iloc[i], errors="coerce")):
        log_issue(i, "visit_date", "date_parse_failure", "Visit date must parse using documented format",
                  "review", "Check source format; do not guess a date")

for i in raw.index[raw.duplicated(keep=False)]:
    log_issue(i, "*row*", "duplicate_row", "Exact duplicate rows require review",
              "review", "Verify source; do not delete automatically")

if "patient_id" in raw.columns:
    dup = raw["patient_id"].notna() & raw["patient_id"].duplicated(keep=False)
    for i in raw.index[dup]:
        log_issue(i, "patient_id", "duplicate_patient_id",
                  "Repeated patient identifiers require review; repeat encounters may be valid",
                  "high", "Confirm record meaning before any merge or deletion")

exception_log = pd.DataFrame(exceptions)
display(exception_log)
exception_log.to_csv(OUT / "exception_log.csv", index=False)


## 4. Numeric summaries — not clinical diagnosis

In [ ]:
numeric = df.select_dtypes(include="number")
display(numeric.describe().T)
print("No clinical range rules are applied because no approved clinical reference range was supplied.")


## 5. Save outputs

In [ ]:
df.to_csv(OUT / "cleaned_patient_records.csv", index=False)
print("Saved cleaned_patient_records.csv")
print("Saved exception_log.csv")
print("Saved quality_profile.csv")


## 6. Final review
- Confirm the dataset remains labelled synthetic demonstration data.
- Complete the field dictionary using the task's governance pack if it becomes available.
- Review duplicate IDs; do not assume they are erroneous.
- Do not invent missing clinical values.
- Do not publish real or identifiable patient data.
